# Probability calibration and threshold optimisation

Run from top to bottom. Quick mode is a patient subset, not final results. Notebook outputs are separate from production reports.

In [ ]:
from pathlib import Path
import sys, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src'/'features.py').exists()), None)
assert ROOT is not None, 'Open the notebook from the project repository'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.notebook_utils import *
QUICK = True  # False uses the entire eligible dataset; keep consistent across notebooks
OUT = ROOT/'reports'/('notebook_quick' if QUICK else 'notebook_full')
OUT.mkdir(parents=True, exist_ok=True)


Load partitions

In [ ]:
raw, x, y, g, parts = load_cohort(ROOT, QUICK)
train, cal, tune, test = (parts[k] for k in ("train","calibration","threshold","test"))
X, Y, G = x.iloc[train], y.iloc[train], g.iloc[train]
display(split_table(y,g,parts))

Fit XGBoost, then fit sigmoid calibration on a separate partition

In [ ]:
from src.models.boosting import models
from src.evaluation.calibration import calibrate,apply,threshold
m=make_model(X,models()['XGBoost'],quick=QUICK).fit(X,Y)
cm=calibrate(m.predict_proba(x.iloc[cal])[:,1],y.iloc[cal]); vp=apply(cm,m.predict_proba(x.iloc[tune])[:,1]); cut=threshold(y.iloc[tune],vp)
print('Validation-selected F1 threshold:',cut)

Assess fixed choices on test; do not tune using these results.

In [ ]:
from sklearn.calibration import calibration_curve
rp=m.predict_proba(x.iloc[test])[:,1]; cp=apply(cm,rp)
r=pd.DataFrame([{'mode':name,**scores(y.iloc[test],p,t)} for name,p,t in [('raw',rp,.5),('calibrated',cp,.5),('optimized',cp,cut)] ]); display(r); r.to_csv(OUT/'calibration_metrics.csv',index=False)
for name,p in [('raw',rp),('calibrated',cp)]:
    obs,pred=calibration_curve(y.iloc[test],p,n_bins=8,strategy='quantile'); plt.plot(pred,obs,'o-',label=name)
plt.plot([0,1],[0,1],'--'); plt.xlabel('Predicted probability'); plt.ylabel('Observed fraction'); plt.legend(); plt.show()